## Zadanie 1

In [1]:
import java.time.*
import kotlin.random.*

enum class CostType(val costType: String) {
    REFUELING("Tankowanie"),
    SERVICE("Serwis"),
    PARKING("Parking"),
    INSURANCE("Ubezpieczenie"),
    TICKET("Mandat")
}

data class Cost (
    val type: CostType,
    val date: LocalDate,
    val amount: Int
)

object DataProvider {
    val generalCosts = List(5) {
        Cost(
            CostType
                .values()[Random.nextInt(CostType.values().size)],
            LocalDate.of(
                2025,
                Random.nextInt(1,13),
                Random.nextInt(1,28)),
            Random.nextInt(5000)
        )
    }
}

In [2]:
fun groupedByCost(lst: List<Cost>): Map<Month, List<Cost>> = lst
    .groupBy { it.date.month }
    .toSortedMap()
    .mapValues { entry ->
        entry.value.sortedBy { it.amount }
    }

println(groupedByCost(DataProvider.generalCosts))

{MARCH=[Cost(type=INSURANCE, date=2025-03-10, amount=742), Cost(type=INSURANCE, date=2025-03-20, amount=4175)], JUNE=[Cost(type=PARKING, date=2025-06-09, amount=539)], AUGUST=[Cost(type=TICKET, date=2025-08-09, amount=2084)], NOVEMBER=[Cost(type=REFUELING, date=2025-11-27, amount=1056)]}


## Zadanie 2

In [3]:
fun printCosts(lst: List<Cost>) {
    val groupedByMonth = lst.groupBy { it.date.month }
        .toSortedMap()

    groupedByMonth.forEach { (month, costs) ->
        println(month.name)

        costs.forEach { cost ->
            println("${cost.date.dayOfMonth} ${cost.type.costType.uppercase()} ${cost.amount}")
        }
        println()
    }
}

printCosts(DataProvider.generalCosts)

MARCH
10 UBEZPIECZENIE 742
20 UBEZPIECZENIE 4175

JUNE
9 PARKING 539

AUGUST
9 MANDAT 2084

NOVEMBER
27 TANKOWANIE 1056



## Zadanie 3

In [4]:
sealed class monthlyCostStatus() {
    object NoCosts : monthlyCostStatus()

    data class withinLimit(val total: Int) : monthlyCostStatus()

    data class overLimit(val total: Int, val exceededBy: Int) : monthlyCostStatus()
}

fun classifyMonthlyCosts(costs: List<Cost>, month: Month, limit: Int): monthlyCostStatus = costs
    .filter { it.date.month == month }
    .sumOf { it.amount }
    .let { total ->
        if (total == 0) {
            monthlyCostStatus.NoCosts
        } else if (total <= limit) {
            monthlyCostStatus.withinLimit(total)
        }
        else {
            monthlyCostStatus.overLimit(total, total - limit)
        }
    }

println(classifyMonthlyCosts(DataProvider.generalCosts, Month.JANUARY, 1000))

Line_6_jupyter$monthlyCostStatus$NoCosts@5b53302f


## Zadanie 4

In [5]:
interface costFormatter {
    fun format(cost: Cost): String
}

object PlCostFormatter : costFormatter {
    override fun format(cost: Cost): String = "${cost.date.dayOfMonth} ${cost.type.costType.uppercase()} ${cost.amount}Zł"
}

fun formatCosts(costs: List<Cost>, formatter: costFormatter): String = costs.sortedBy { it.date.dayOfMonth } .joinToString("\n") { formatter.format(it) }

print(formatCosts(DataProvider.generalCosts, PlCostFormatter))

9 PARKING 539Zł
9 MANDAT 2084Zł
10 UBEZPIECZENIE 742Zł
20 UBEZPIECZENIE 4175Zł
27 TANKOWANIE 1056Zł